# Analysis: AlphaFold2 (ColabFold) vs ESMFold on 6 CASP15 FM targets

This notebook rebuilds every table and figure from `results/` only (plus `config/targets.yaml`).
It calls the same functions as `python scripts/analyze.py`, one cell per table or figure.

Run it from the repo with the `envs/p3_tools` kernel. Nothing here reads the native structures:
Figure 4 uses the superposed copies the evaluate step wrote to `results/<target>/superposition/`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'scripts'))

import pandas as pd
from IPython.display import Image, Markdown, display

import analyze

d = analyze.load()
pd.set_option('display.precision', 3)
d['runs'].head()

## All runs
One row per run (`results/tables/all_runs.csv`). Duplicate depth runs (`is_duplicate_depth`) repeat the full MSA and are left out of the depth analysis.

In [ ]:
d['runs']

## Q1. At what MSA size does AF2 drop to ESMFold's accuracy?
Smallest MSA (actual sequences used, and Neff/L) where AF2's TM-score is at or above ESMFold's.

In [ ]:
analyze.table_smallest_msa(d)

In [ ]:
display(Image(filename=analyze.fig1(d)))
display(Image(filename=analyze.fig1b(d)))

### T1122 (near-single-sequence case): all 5 AF2 models, full MSA vs depth 1
The full MSA is the query plus an identical copy. If the gap between the two runs comes from ranking luck, the 5 models should overlap; if the duplicate row changes AF2's behaviour, they should not.

In [ ]:
t = analyze.table_t1122_models(d)
if t is None:
    print('Not available yet: run the T1122 all-model reruns and the evaluate step first.')
else:
    display(t)
    display(Image(filename=analyze.figS1(d)))

## Q2. Does ESM-2 perplexity predict which targets are hard?
Spearman correlations with exact permutation p-values. **n = 6 targets: |rho| >= 0.89 is needed for p < 0.05**, so read these as hints.

In [ ]:
display(analyze.table_perplexity(d))
display(analyze.table_spearman(d))
display(Image(filename=analyze.fig2(d)))

## The chain: perplexity, MSA depth and accuracy
ESM-2 learned from UniRef, so proteins with many relatives get low perplexity, and those same relatives fill AF2's MSA. Figure 7 tests perplexity vs MSA depth (T1147, the positive control, in gray and left out of the statistics); Figure 8 shows TM-score vs perplexity with point size = Neff/L. `af2_plateau` shows how much of AF2's MSA gain is reached with 8 or fewer sequences.

In [ ]:
display(analyze.table_plateau(d))
display(Image(filename=analyze.fig7(d)))
display(Image(filename=analyze.fig8(d)))
display(Markdown(analyze.chain_md(d)))

## Is each model confident only when it is right?
Confident = mean pLDDT >= 70 over the FM domain; right = TM-score >= 0.5.

In [ ]:
display(analyze.table_confidence_quadrants(d))
display(Image(filename=analyze.fig3(d)))

## Structures and PAE maps

In [ ]:
display(Image(filename=analyze.fig4(d)))

In [ ]:
display(Image(filename=analyze.fig5(d)))

## Q3. Accuracy vs speed
AF2 time per model is shown next to ESMFold (1 model). AF2 MSA search times may be cached by the ColabFold server (see the summary).

In [ ]:
display(analyze.table_speed(d))
display(Image(filename=analyze.fig6(d)))

## Summary for the README
The same markdown block that `analyze.py` writes to `results/summary.md`.

In [ ]:
display(Markdown(analyze.summary_md(d)))